# Veri üretimi · Gemma 4 12B · Colab **ve** Kaggle
Aynı defter iki yerde çalışır; ikisi **aynı anda** açılabilir. Önce **kart**, sonra hikâye üretir. Filtre henüz yok (görev S1); ham dosyalar sonra filtreden geçirilir.

**Önce:** `yz/veri-seti` değişiklikleri AI ve Docs'ta `main`'e birleştirilmiş olmalı (defter `main`'i indirir).

**Colab:** Çalışma zamanı → **T4 GPU** → 1. hücreye adını yaz → *Tümünü çalıştır*. Dosyalar Drive'a (`kaydir_veri/`) gider; koparsa yeniden çalıştır, **kaldığı yerden devam eder** (yarım kalan son satır otomatik silinir).
İstersen Hugging Face token'ını Colab'ın sol kenarındaki 🔑 **Secrets**'a `HF_TOKEN` adıyla ekle; yoksa defter sorar.

**Kaggle:** *File → Import Notebook* → bu dosya. Sağ panel: **Accelerator: GPU T4 x2**, **Internet: On** (telefon onayı gerekir). *Add-ons → Secrets* → `HF_TOKEN` ekle (gece modunda zorunlu).
- Ekran başındaysan *Run All*; bitince sağdaki **Output**'tan `kaydir_veri.zip`'i indir (oturum kapanınca dosyalar silinir).
- Gece çalışsın: **Save Version → Save & Run All (Commit)** → arka planda çalışır; 10,5 saatte kendini durdurur ve zip'i hazırlar. Bitince o sürümün Output'undan zip'i indir.

Model lisansı: huggingface.co/google/gemma-4-12B-it sayfasında kabul edilmiş hesap. Gemma 4 Apache 2.0: çıktısıyla eğitim serbest.

## 1 · Ayar

In [ ]:
KISI = ''              # kendi adın; her hesapta FARKLI olsun ('mert', 'sokul', 'kara'...). Aynı ad = aynı promptlar = boşa giden GPU saati
KART_ADET = 1500       # bu dosyada toplam kaç kart (Colab'da ertesi gün artırıp devam edebilirsin)
HIKAYE_ADET = 4500     # kaç hikâye
DAL = 'main'           # AI ve Docs depolarının dalı
SURE_SAAT = 10.5       # bu oturumda en fazla kaç saat üretilsin (Kaggle'ın 12 saat sınırına takılmasın)
KART_PAYI = 0.25       # önce kart: sürenin %25'i karta, kalanı hikâyeye (11 Kasım: hikâye:kart ≈ 2:1). Sadece kart için HIKAYE_ADET = 0 ve KART_PAYI = 1.0

## 2 · Hazırlık: depoları indir, klasörleri aç

In [ ]:
import os, sys, time, subprocess
assert KISI, "1. hücrede KISI'ya adını yaz ('mert' / 'sokul')"
T0 = time.time()                         # süre buradan sayılır (kurulum + model yükleme dahil)
KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))   # Colab'da da /kaggle klasörü olabiliyor; ortam değişkenine bak
ORTAM = 'kaggle' if KAGGLE else 'colab'
KOK = '/kaggle/working' if KAGGLE else '/content'
if KAGGLE:
    VERI = f'{KOK}/kaydir_veri'
    OTURUM = globals().get('OTURUM') or time.strftime('%m%d-%H%M')   # Kaggle'da her oturum yeni dosya; hücre tekrar çalışınca değişmez
else:
    from google.colab import drive; drive.mount('/content/drive')
    VERI = '/content/drive/MyDrive/kaydir_veri'
    OTURUM = time.strftime('%m%d')                                     # Colab'da her gün yeni dosya ve yeni tohum; aynı gün tekrar çalışırsa kaldığı yerden devam eder
for depo in ('AI', 'Docs'):           # depolar herkese açık; varsa güncellenir, silinmez
    yol = f'/tmp/{depo}'
    if os.path.isdir(f'{yol}/.git'):
        subprocess.run(['git', '-C', yol, 'fetch', '-q', '--depth', '1', 'origin', DAL], check=True)
        subprocess.run(['git', '-C', yol, 'checkout', '-q', '-B', DAL, 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', DAL, f'https://github.com/IleriProgramlamaProje/{depo}', yol], check=True)
for ad in [m for m in sys.modules if m == 'ai' or m.startswith('ai.')]:
    del sys.modules[ad]                # güncellenen kod yeniden yüklensin
if '/tmp/AI' not in sys.path:
    sys.path.insert(0, '/tmp/AI')      # Docs kardeş klasörde: tema.json oradan okunur
os.makedirs(f'{VERI}/ham', exist_ok=True); os.makedirs(f'{VERI}/temiz', exist_ok=True)
print(ORTAM, '·', VERI, '·', subprocess.run(['git', '-C', '/tmp/AI', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

## 3 · Kurulum ve Hugging Face girişi

In [ ]:
import getpass
if 'motor' not in globals():          # model yüklendikten sonra paket güncellemesi yapma (sürümler karışır)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'bitsandbytes'], check=True)
token = None
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata; token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if KAGGLE and not token and os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch':
    raise RuntimeError("Gece modu için Add-ons → Secrets → HF_TOKEN ekle")
from huggingface_hub import login
login(token=(token or getpass.getpass('Hugging Face token: ')).strip())   # yapıştırırken gelen boşluk 401 verdirmesin

## 4 · Modeli aç (5–10 dk)
Kaggle'da 2 GPU: 8-bit (daha temiz yazım). Colab'da 1 GPU: 4-bit. Bozuk kelimeleri sonra filtre ayıklayacak. Hücre tekrar çalışırsa model ikinci kez yüklenmez.

In [ ]:
import torch
from ai.veri_uretimi.uretici import motor_ac, uret
MODEL = 'google/gemma-4-12B-it'
iki_gpu = torch.cuda.device_count() >= 2
if 'motor' not in globals():
    motor, MOTOR_ADI = motor_ac('transformers8bit' if iki_gpu else 'transformers4bit', MODEL)
GRUP = 8 if iki_gpu else 4
print('Motor:', MOTOR_ADI, '· GPU:', torch.cuda.device_count())

## 5 · Kart üret
Süre her grupta kontrol edilir: sürenin %25'ı dolunca karttan hikâyeye geçer, kaldığı yer dosyada. Ekranda hız yazar (saatte kaç kart), o satırı gruba at.

In [ ]:
BITIS = {'kart': T0 + SURE_SAAT * KART_PAYI * 3600, 'hikaye': T0 + SURE_SAAT * 3600}
HAM_K = f'{VERI}/ham/kart_{KISI}_{ORTAM}_{OTURUM}.jsonl'
HAM_H = f'{VERI}/ham/hikaye_{KISI}_{ORTAM}_{OTURUM}.jsonl'
def paketle():                      # Kaggle'da hata olsa da o ana kadar üretilen dosyalar zip'e girsin
    if KAGGLE:
        import shutil; shutil.make_archive(f'{KOK}/kaydir_veri', 'zip', VERI); print('Output → kaydir_veri.zip')
def parca_parca(tur, adet, yol):     # süre her grupta kontrol edilir; dolunca durur, kalan iş dosyada
    try:
        uret(tur, adet, yol, motor_adi=MOTOR_ADI, grup=GRUP, tohum=f'{KISI}-{ORTAM}-{OTURUM}-{tur}',
             model=MODEL, motor=motor, tolerans=True, onar=True, bitis=BITIS[tur])
    except Exception:
        paketle(); raise
parca_parca('kart', KART_ADET, HAM_K)

## 6 · Hikâye üret

In [ ]:
parca_parca('hikaye', HIKAYE_ADET, HAM_H)

## 7 · Paket
Ham dosyalar `ham/` klasöründe. Filtre (görev S1) hazır olunca `temiz/`'e süzülür; eğitimde sadece `temiz/` kullanılır.

In [ ]:
print('Kart:', HAM_K); print('Hikâye:', HAM_H)
paketle()

## Bitince
Colab: *Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil* (birim harcamasın). Ertesi gün adetleri artır (1500 → 3000), tümünü çalıştır: kaldığı yerden devam eder.